In [1]:

# ============================================================
# CIFAR-10 RESNET CHALLENGE - IMPROVED TRAINING
# Complete standalone Kaggle cell
# ============================================================

import os
import re
import math
import time
import copy
import random
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
EPOCHS = 200
BATCH_SIZE = 128
NUM_WORKERS = 2

LEARNING_RATE = 0.1
WEIGHT_DECAY = 5e-4
EMA_DECAY = 0.999

MIX_PROBABILITY = 0.40
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 0.5
CLEAN_LAST_EPOCHS = 15

SAVE_EVERY = 10

OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if DEVICE.type != "cuda":
    raise RuntimeError(
        "Enable a Kaggle GPU before running this notebook."
    )

MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.backends.cudnn.benchmark = True

print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. FIND COMPETITION DATA
# ============================================================

def find_dataset():
    for csv_path in Path("/kaggle/input").rglob(
        "train_labels.csv"
    ):
        root = csv_path.parent

        if (
            (root / "train_images").is_dir()
            and (root / "test_images").is_dir()
            and (root / "sample_submission.csv").exists()
        ):
            return root

    raise FileNotFoundError(
        "Competition data not found. "
        "Attach the CIFAR-10 competition dataset."
    )

ROOT = find_dataset()

train_df = pd.read_csv(ROOT / "train_labels.csv")
sample_df = pd.read_csv(ROOT / "sample_submission.csv")

assert {"id", "label"}.issubset(train_df.columns)
assert {"id", "label"}.issubset(sample_df.columns)

classes = sorted(train_df["label"].unique().tolist())

assert len(classes) == 10

class_to_idx = {
    label: i for i, label in enumerate(classes)
}

idx_to_class = {
    i: label for label, i in class_to_idx.items()
}

labels = np.array(
    [class_to_idx[x] for x in train_df["label"]],
    dtype=np.int64
)

print("Dataset:", ROOT)
print("Train images:", len(train_df))
print("Test images:", len(sample_df))

# ============================================================
# 3. MATCH IMAGE FILES
# ============================================================

def match_images(folder, ids):
    folder = Path(folder)

    files = [
        p for p in folder.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {
            ".png", ".jpg", ".jpeg", ".webp", ".bmp"
        }
    ]

    mapping = {}

    for path in files:
        numbers = re.findall(r"\d+", path.stem)

        if len(numbers) == 1:
            number = int(numbers[0])

            if number in mapping:
                raise ValueError(
                    f"Duplicate image ID: {number}"
                )

            mapping[number] = path

    matched = []

    for image_id in ids:
        number = int(image_id)

        if number not in mapping:
            raise FileNotFoundError(
                f"Missing image ID {number} in {folder}"
            )

        matched.append(mapping[number])

    if len(set(matched)) != len(matched):
        raise ValueError("Duplicate image mapping detected.")

    return matched

train_paths = match_images(
    ROOT / "train_images",
    train_df["id"].tolist()
)

test_paths = match_images(
    ROOT / "test_images",
    sample_df["id"].tolist()
)

assert len(train_paths) == 50000
assert len(test_paths) == 2000

print("Image matching successful.")

# ============================================================
# 4. DATA AUGMENTATION
# ============================================================

train_transform = T.Compose([
    T.RandomCrop(
        32,
        padding=4,
        padding_mode="reflect"
    ),
    T.RandomHorizontalFlip(),
    T.RandAugment(
        num_ops=2,
        magnitude=7
    ),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    T.RandomErasing(
        p=0.10,
        scale=(0.02, 0.08),
        ratio=(0.5, 2.0),
        value=0
    )
])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

class CIFARDataset(Dataset):
    def __init__(self, paths, targets=None, transform=None):
        self.paths = paths
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as img:
            image = img.convert("RGB")

        image = self.transform(image)

        if self.targets is None:
            return image

        return image, int(self.targets[index])

train_loader = DataLoader(
    CIFARDataset(train_paths, labels, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0
)

test_loader = DataLoader(
    CIFARDataset(test_paths, None, test_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0
)

# ============================================================
# 5. MODEL ARCHITECTURE
# ============================================================

class SEBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        hidden = max(8, channels // 16)

        self.fc1 = nn.Conv2d(channels, hidden, 1)
        self.fc2 = nn.Conv2d(hidden, channels, 1)

    def forward(self, x):
        scale = F.adaptive_avg_pool2d(x, 1)
        scale = F.relu(self.fc1(scale), inplace=True)
        scale = torch.sigmoid(self.fc2(scale))
        return x * scale


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SEBlock(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        x = F.relu(
            self.bn1(self.conv1(x)),
            inplace=True
        )

        x = self.bn2(self.conv2(x))
        x = self.se(x)

        return F.relu(x + identity, inplace=True)


class EfficientResNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(
                3, 64,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.stage1 = self.make_stage(64, 64, 3, 1)
        self.stage2 = self.make_stage(64, 128, 3, 2)
        self.stage3 = self.make_stage(128, 256, 3, 2)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, 10)

        self.apply(self.initialize)

        # Zero-initialize final BN in residual branches.
        # This helps each block begin near an identity mapping.
        for module in self.modules():
            if isinstance(module, ResidualBlock):
                nn.init.zeros_(module.bn2.weight)

    @staticmethod
    def initialize(module):
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(
                module.weight,
                mode="fan_out",
                nonlinearity="relu"
            )

        elif isinstance(module, nn.BatchNorm2d):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    @staticmethod
    def make_stage(cin, cout, blocks, stride):
        layers = [ResidualBlock(cin, cout, stride)]

        for _ in range(1, blocks):
            layers.append(ResidualBlock(cout, cout))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


model = EfficientResNet().to(DEVICE)

PARAM_COUNT = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", f"{PARAM_COUNT:,}")

assert PARAM_COUNT <= 5_000_000
print("Parameter limit: PASSED")

# ============================================================
# 6. MIXUP / CUTMIX
# ============================================================

def mixup_cutmix(images, targets):
    if random.random() >= MIX_PROBABILITY:
        return images, targets, targets, 1.0

    permutation = torch.randperm(
        images.size(0),
        device=images.device
    )

    targets_b = targets[permutation]

    if random.random() < 0.5:
        lam = float(
            np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
        )

        mixed = (
            lam * images
            + (1.0 - lam) * images[permutation]
        )

        return mixed, targets, targets_b, lam

    lam = float(
        np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA)
    )

    _, _, height, width = images.shape

    cut_ratio = math.sqrt(1.0 - lam)
    cut_w = int(width * cut_ratio)
    cut_h = int(height * cut_ratio)

    cx = random.randrange(width)
    cy = random.randrange(height)

    x1 = max(0, cx - cut_w // 2)
    x2 = min(width, cx + cut_w // 2)
    y1 = max(0, cy - cut_h // 2)
    y2 = min(height, cy + cut_h // 2)

    mixed = images.clone()

    mixed[:, :, y1:y2, x1:x2] = (
        images[permutation, :, y1:y2, x1:x2]
    )

    lam = 1.0 - (
        (x2 - x1) * (y2 - y1)
        / float(width * height)
    )

    return mixed, targets, targets_b, lam

# ============================================================
# 7. EMA MODEL
# ============================================================

ema_model = copy.deepcopy(model).eval()

for parameter in ema_model.parameters():
    parameter.requires_grad_(False)

@torch.no_grad()
def update_ema(ema, source, decay):
    for ema_p, model_p in zip(
        ema.parameters(), source.parameters()
    ):
        ema_p.mul_(decay).add_(
            model_p.detach(),
            alpha=1.0 - decay
        )

    # Copy BatchNorm buffers.
    for ema_buffer, model_buffer in zip(
        ema.buffers(), source.buffers()
    ):
        ema_buffer.copy_(model_buffer)

# ============================================================
# 8. OPTIMIZER AND SCHEDULER
# ============================================================

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=0.9,
    weight_decay=WEIGHT_DECAY,
    nesterov=True
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.15,
    anneal_strategy="cos",
    div_factor=10,
    final_div_factor=1000
)

scaler = torch.amp.GradScaler("cuda")

# ============================================================
# 9. TRAINING
# ============================================================

history = []
global_step = 0
start_time = time.time()

print("\nStarting training...")

for epoch in range(EPOCHS):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for images, targets in train_loader:
        images = images.to(
            DEVICE, non_blocking=True
        )

        targets = targets.to(
            DEVICE, non_blocking=True
        )

        if epoch < EPOCHS - CLEAN_LAST_EPOCHS:
            images, ya, yb, lam = mixup_cutmix(
                images, targets
            )
        else:
            ya, yb, lam = targets, targets, 1.0

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(images)

            loss = (
                lam * F.cross_entropy(
                    logits,
                    ya,
                    label_smoothing=0.05
                )
                + (1.0 - lam) * F.cross_entropy(
                    logits,
                    yb,
                    label_smoothing=0.05
                )
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        global_step += 1

        decay = min(
            EMA_DECAY,
            (global_step + 1) / (global_step + 10)
        )

        update_ema(
            ema_model,
            model,
            decay
        )

        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

    epoch_loss = total_loss / total_samples

    history.append({
        "epoch": epoch + 1,
        "loss": epoch_loss,
        "lr": optimizer.param_groups[0]["lr"]
    })

    print(
        f"Epoch {epoch+1:03d}/{EPOCHS} | "
        f"Loss: {epoch_loss:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.6f}",
        flush=True
    )

    pd.DataFrame(history).to_csv(
        OUT / "improved_training_history.csv",
        index=False
    )

    if (
        (epoch + 1) % SAVE_EVERY == 0
        or epoch + 1 == EPOCHS
    ):
        torch.save(
            {
                "epoch": epoch + 1,
                "model": model.state_dict(),
                "ema": ema_model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "global_step": global_step
            },
            OUT / "improved_recovery.pth"
        )

print(
    "\nTraining time (minutes):",
    round((time.time() - start_time) / 60, 2)
)

torch.save(
    ema_model.state_dict(),
    OUT / "improved_final_model.pth"
)

# ============================================================
# 10. TEST-TIME AUGMENTATION
# ============================================================

ema_model.eval()

@torch.inference_mode()
def predict_logits(images):
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        logits = ema_model(images)

    return logits.float()

def shift_normalized(images, dx, dy):
    padded = F.pad(
        images,
        (2, 2, 2, 2),
        mode="reflect"
    )

    top = 2 + dy
    left = 2 + dx

    return padded[
        :, :,
        top:top+32,
        left:left+32
    ]

baseline_predictions = []
tta_predictions = []

print("\nGenerating predictions...")

for images in test_loader:
    images = images.to(
        DEVICE,
        non_blocking=True
    )

    original = predict_logits(images)
    flipped = predict_logits(images.flip(-1))

    # Original method: average logits.
    baseline_logits = (
        original + flipped
    ) / 2

    baseline_probs = baseline_logits.softmax(dim=1)

    # Additional translation views.
    views = [
        shift_normalized(images, -1, 0),
        shift_normalized(images, 1, 0),
        shift_normalized(images, 0, -1),
        shift_normalized(images, 0, 1)
    ]

    shift_probs = [
        predict_logits(view).softmax(dim=1)
        for view in views
    ]

    # Conservative blend with original TTA.
    augmented_probs = (
        0.75 * baseline_probs
        + 0.25 * (
            sum(shift_probs) / len(shift_probs)
        )
    )

    baseline_predictions.extend(
        baseline_probs.argmax(dim=1).cpu().tolist()
    )

    tta_predictions.extend(
        augmented_probs.argmax(dim=1).cpu().tolist()
    )

# ============================================================
# 11. SAVE SUBMISSIONS
# ============================================================

def save_submission(indices, filename):
    official_labels = [
        idx_to_class[int(i)] for i in indices
    ]

    submission = pd.DataFrame({
        "id": sample_df["id"].to_numpy(),
        "label": official_labels
    })

    assert len(submission) == 2000
    assert submission["id"].is_unique
    assert not submission.isna().any().any()

    path = OUT / filename
    submission.to_csv(path, index=False)

    print("Saved:", path)
    return submission

baseline_submission = save_submission(
    baseline_predictions,
    "submission_improved_baseline.csv"
)

tta_submission = save_submission(
    tta_predictions,
    "submission_improved_tta.csv"
)

# Standard Kaggle submission filename.
tta_submission.to_csv(
    OUT / "submission.csv",
    index=False
)

changed = sum(
    a != b for a, b in zip(
        baseline_predictions,
        tta_predictions
    )
)

print("\n" + "=" * 60)
print("TRAINING AND SUBMISSION COMPLETE")
print("=" * 60)
print("Parameters:", PARAM_COUNT)
print("Predictions changed by TTA:", changed)
print("Standard submission:", OUT / "submission.csv")
print("Model:", OUT / "improved_final_model.pth")
print("Done!")


GPU: Tesla T4
Dataset: /kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl
Train images: 50000
Test images: 2000
Image matching successful.
Trainable parameters: 4,362,986
Parameter limit: PASSED

Starting training...
Epoch 001/200 | Loss: 2.0452 | LR: 0.010247
Epoch 002/200 | Loss: 1.9090 | LR: 0.010984
Epoch 003/200 | Loss: 1.7899 | LR: 0.012203
Epoch 004/200 | Loss: 1.6533 | LR: 0.013891
Epoch 005/200 | Loss: 1.5349 | LR: 0.016030
Epoch 006/200 | Loss: 1.4266 | LR: 0.018596
Epoch 007/200 | Loss: 1.3360 | LR: 0.021560
Epoch 008/200 | Loss: 1.2752 | LR: 0.024892
Epoch 009/200 | Loss: 1.2311 | LR: 0.028553
Epoch 010/200 | Loss: 1.1603 | LR: 0.032503
Epoch 011/200 | Loss: 1.1090 | LR: 0.036701
Epoch 012/200 | Loss: 1.0889 | LR: 0.041099
Epoch 013/200 | Loss: 1.0885 | LR: 0.045649
Epoch 014/200 | Loss: 1.0593 | LR: 0.050302
Epoch 015/200 | Loss: 0.9905 | LR: 0.055006
Epoch 016/200 | Loss: 1.0061 | LR: 0.059710
Epoch 017/200 | Loss: 1.0039 | LR: 0.064363
Epoch 018/200 | 